### Problem 001: Kth Largest Element in a Stream (LeetCode 703)

### Problem Definition and Constraints
Design a class to find the $k$-th largest integer in a stream of values, including duplicates. The stream is not necessarily sorted.
Implement the `KthLargest` class:
* `KthLargest(int k, int[] nums)` Initializes the object with the integer `k` and the stream of integers `nums`.
* `int add(int val)` Appends the integer `val` to the stream and returns the $k$-th largest element in the stream.

* Constraints:
  * 1 <= k <= 10^4
  * 0 <= nums.length <= 10^4
  * -10^4 <= nums[i], val <= 10^4
  * There will always be at least `k` integers in the stream when you search for the $k$-th integer.

### Brute Force Approach
Every time a new number is added via `add(val)`, we append it to a standard array and then sort the entire array in descending order. Once sorted, we simply return the element at index `k - 1`.
* Time Complexity: $O(m \cdot n \log n)$ — Where $m$ is the number of `add` calls and $n$ is the total elements. Sorting the entire list on every single insertion is extremely slow.
* Space Complexity: $O(n)$ — We store every single number that ever gets added to the stream.

### Optimized Approach (Min-Heap)
To achieve $O(\log k)$ insertions, we use a Min-Heap. By strictly limiting the size of the heap to exactly `k` elements, the heap will naturally hold only the `k` largest numbers seen so far. Because it is a *Min*-Heap, the smallest number out of those `k` largest numbers will always sit at the very top (index 0). When `add()` is called, we push the new value onto the heap. If the heap's size exceeds `k`, we immediately pop the top element (which removes the smallest number, kicking out the "poorest" VIP). We then return the new top element.
* Time Complexity: $O(n \log k)$ for initialization, and $O(\log k)$ for each `add()` call. Pushing and popping from a heap of size $k$ takes logarithmic time relative to $k$.
* Space Complexity: $O(k)$ — We permanently discard any numbers that aren't in the top `k`, so our memory footprint never grows beyond size `k`.

In [1]:
import heapq
from typing import List

class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        # Store k so we know the maximum size of our VIP club
        self.k = k
        self.minHeap = nums
        
        # heapq.heapify transforms a standard list into a Min-Heap in-place in O(n) time
        heapq.heapify(self.minHeap)
        
        # If the starting array has more than k elements, pop the smallest ones 
        # until we are down to exactly k elements.
        while len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)

    def add(self, val: int) -> int:
        # 1. Push the new person into the club
        heapq.heappush(self.minHeap, val)
        
        # 2. If the club has more than k people, kick out the poorest one
        if len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)
            
        # 3. The poorest person in the VIP club (the root of the min-heap) 
        # is the k-th largest element overall.
        return self.minHeap[0]

# Your KthLargest object will be instantiated and called as such:
# obj = KthLargest(k, nums)
# param_1 = obj.add(val)


### Problem 002: Last Stone Weight (LeetCode 1046)

### Problem Definition and Constraints
You are given an array of integers `stones` where `stones[i]` represents the weight of the $i$-th stone.
We repeatedly choose the two heaviest stones and smash them together. 
* If `x == y`, both stones are destroyed.
* If `x < y`, the stone of weight `x` is destroyed, and the stone of weight `y` has a new weight of `y - x`.
Return the weight of the last remaining stone or return 0 if none remain.

* Constraints:
  * 1 <= stones.length <= 30
  * 1 <= stones[i] <= 1000

### Examples
* **Example 1:**
  * Input: `stones = [2,7,4,1,8,1]`
  * Output: `1`
  * Explanation: 
    * Smash 8 and 7 -> 1. Array becomes `[2,4,1,1,1]`
    * Smash 4 and 2 -> 2. Array becomes `[2,1,1,1]`
    * Smash 2 and 1 -> 1. Array becomes `[1,1,1]`
    * Smash 1 and 1 -> 0. Array becomes `[1]`
    * Last stone is 1.

### Brute Force Approach
The naive approach is to use a standard array. Inside a `while` loop, we sort the entire array in descending order, pop the first two elements, calculate their difference, and append the result back to the array. We repeat this until the array length is 1 or 0.
* Time Complexity: $O(n^2 \log n)$ — We have to re-sort the entire array of size $n$ every single time we do a smash operation (which happens roughly $n$ times).
* Space Complexity: $O(1)$ or $O(n)$ depending on if the sorting is done in-place.

### Optimized Approach (Max-Heap Simulation)
To avoid resorting the whole array every time, we use a Max-Heap. Because Python's `heapq` library only supports Min-Heaps, we first negate all values in the array (e.g., `5` becomes `-5`). The "heaviest" stone becomes the "smallest" negative number, naturally bubbling to the top of the Min-Heap.
While there is more than 1 stone in the heap, we pop the top two elements (multiplying by -1 to get their true positive weights back). If the first is heavier than the second, we calculate `first - second`, negate it, and push it back into the heap. If they are equal, we push nothing. We return the final stone (made positive again) or `0` if the heap is empty.
* Time Complexity: $O(n \log n)$ — `heapify` takes $O(n)$. We then do at most $n$ smashes. Each smash requires two `heappop` and one `heappush` operations, taking $O(\log n)$ time each.
* Space Complexity: $O(n)$ — We store the negated weights in a heap structure of size $n$.


In [2]:
import heapq
from typing import List

class Solution:
    def lastStoneWeight(self, stones: List[int]) -> int:
        
        # 1. Transform into a Max-Heap using the "Negative Trick"
        # We multiply every stone by -1 so the heaviest stones become the smallest numbers
        max_heap = [-s for s in stones]
        heapq.heapify(max_heap)
        
        # 2. Simulate the gladiator arena
        # We need at least 2 stones to have a fight
        while len(max_heap) > 1:
            
            # Pop the two "smallest" (most negative) numbers and make them positive again
            first = -heapq.heappop(max_heap)   # The absolute heaviest stone
            second = -heapq.heappop(max_heap)  # The second heaviest stone
            
            # If the first is bigger, there is a leftover piece.
            if first > second:
                leftover = first - second
                # Push the leftover back into the ring (remember to make it negative!)
                heapq.heappush(max_heap, -leftover)
                
            # If first == second, both are destroyed. We do nothing and loop again.
            
        # 3. Check the aftermath
        # If there is a survivor, make it positive and return it.
        if max_heap:
            return -max_heap[0]
            
        # If they completely wiped each other out, return 0.
        return 0

### Problem 003: K Closest Points to Origin (LeetCode 973)

### Problem Definition and Constraints
You are given an array of coordinates `points` and an integer `k`. You need to return the `k` points that are closest to the origin (0, 0).
* The distance is measured using the standard Euclidean formula: $\sqrt{x^2 + y^2}$
* You can return the answer in any order.

* Constraints:
  * 1 <= k <= points.length <= 1000
  * -100 <= points[i][0], points[i][1] <= 100

### Brute Force Approach
Calculate the exact distance for every single point. Save them all in an array, and then fully sort the array from smallest distance to largest distance. Finally, slice off the first `k` elements.
* **Time Complexity:** $O(N \log N)$ — Sorting the entire list of $N$ points.
* **Space Complexity:** $O(N)$ — Storing all $N$ distances in memory.

### Optimized Approach (Max-Heap)
We combine the "VIP Club" and the "Negative Trick" templates!
1. **The Math Shortcut:** Computing square roots is slow. Since we only care about *comparing* distances, we can just use $x^2 + y^2$. If $A^2 < B^2$, then $A < B$.
2. **The VIP Club:** We want to keep a heap of size exactly `k` that holds the *smallest* distances. If we get $k + 1$ elements, we must pop the *largest* distance.
3. **The Negative Trick:** To make the largest distance pop out easily, we need a Max-Heap. We multiply our distance by `-1` before pushing it into Python's Min-Heap.
4. **The Heap Storage:** A heap can hold more than just numbers! We can push a Python "tuple" (a mini list) containing `(-distance, x, y)`. Python will automatically sort the heap using the first item in the tuple (the `-distance`).

* **Time Complexity:** $O(N \log k)$ — We process all $N$ points, but our heap operations only take $\log k$ time because the heap never grows larger than $k$.
* **Space Complexity:** $O(k)$ — We strictly cap the heap at size $k$, saving massive amounts of memory.

In [3]:
import heapq
from typing import List

class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        max_heap = []
        
        for x, y in points:
            # 1. Calculate the distance (ignoring the square root for speed)
            # 2. Apply the Negative Trick so the BIGGEST distance acts as the "smallest" value
            dist = -1 * ((x ** 2) + (y ** 2))
            
            # 3. Push a tuple into the heap: (negative_distance, x-coord, y-coord)
            # Python will use the first item (dist) to decide who bubbles to the top
            heapq.heappush(max_heap, (dist, x, y))
            
            # 4. VIP Club Rule: If the heap exceeds size k, kick out the top element.
            # Because of the Negative Trick, the top element is the FARTHEST point!
            if len(max_heap) > k:
                heapq.heappop(max_heap)
                
        # 5. Compile the final list of surviving VIPs
        result = []
        for dist, x, y in max_heap:
            result.append([x, y])
            
        return result

### Problem 004: Kth Largest Element in an Array (LeetCode 215)

### Problem Definition and Constraints
Given an unsorted array of integers `nums` and an integer `k`, return the `k`-th largest element in the array. 
*Note: It is the k-th largest in sorted order, not the k-th distinct element (duplicates count).*

* Constraints:
  * 1 <= k <= nums.length <= 10^5
  * -10^4 <= nums[i] <= 10^4

### Brute Force Approach
Just use Python's built-in sort function to sort the array in descending order, then return the element at index `k - 1`. 
* **Time Complexity:** $O(N \log N)$ — Sorting the entire array is slow.
* **Space Complexity:** $O(1)$ or $O(N)$ — Depending on whether the sorting is done in-place or creates a new array.

### Optimized Approach (Min-Heap)
We use the exact same **"VIP Club"** template. We iterate through the array, pushing every number into a Min-Heap. The moment the heap size exceeds `k`, we pop the top element (which is the smallest number currently in the heap). 
By the time the loop finishes, the heap will hold exactly the `k` largest numbers from the array. Because it is a Min-Heap, the smallest of those `k` numbers (which is the actual `k`-th largest overall) will be sitting perfectly at the top (`min_heap[0]`).

* **Time Complexity:** $O(N \log k)$ — We process all $N$ elements, but pushing/popping only takes $\log k$ time because the heap size is strictly capped at $k$.
* **Space Complexity:** $O(k)$ — Our heap only ever stores exactly $k$ elements, discarding the rest.

In [4]:
import heapq
from typing import List

class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        min_heap = []
        
        for num in nums:
            # 1. Let the new person into the club
            heapq.heappush(min_heap, num)
            
            # 2. If the club exceeds capacity, kick out the "poorest" member
            if len(min_heap) > k:
                heapq.heappop(min_heap)
                
        # 3. The poorest member of the surviving VIP club is the k-th largest element!
        return min_heap[0]

### Problem 005: Task Scheduler (LeetCode 621)

### Problem Definition and Constraints
You are given an array of CPU `tasks` (represented by letters A-Z) and a cooldown period `n`. 
* Each task takes exactly 1 cycle to run.
* You can run tasks in any order, but if you run a task (like 'A'), you cannot run another 'A' until exactly `n` cycles have passed. 
* During cooldowns, you can run other tasks. If no other tasks are available, the CPU sits "Idle".
Return the minimum total cycles needed to finish all tasks.

* Constraints:
  * 1 <= tasks.length <= 10^4
  * 0 <= n <= 100

### Brute Force Approach
Generate every single possible valid ordering of the tasks and simulate them to see which one takes the least amount of time. 
* **Time Complexity:** Exponential $O(K!)$ — Testing every permutation of tasks is incredibly slow and will result in a Time Limit Exceeded (TLE) error.
* **Space Complexity:** $O(K)$ — For the recursion call stack.

### Optimized Approach (Max-Heap + Cooldown Queue)
We use a **Greedy strategy**: always process the most frequent available task first. 
1. **Count Frequencies:** Count how many times each task appears. We don't actually care *which* letter it is, only its frequency.
2. **The Max-Heap:** Push all the frequencies into a Max-Heap (using the Negative Trick). This ensures the task with the highest remaining frequency is always at the top.
3. **The Cooldown Queue:** When a task runs, we subtract 1 from its frequency. If it still has tasks left to run, it goes into a `queue` (the cooldown room) along with the exact exact `time` it is allowed to come out.
4. **The Simulation:** A global `time` counter ticks up by 1 every loop. We pop from the Max-Heap to run a task. Then, we check the Queue. If the guy at the front of the Queue is done cooling down, we pop him out and push him back into the Max-Heap!

* **Time Complexity:** $O(M)$ — Where $M$ is the total number of tasks. Pushing and popping from the heap technically takes $O(\log 26)$, but since 26 (the alphabet) is a constant limit, it simplifies to $O(1)$ time per task!
* **Space Complexity:** $O(1)$ — The heap and the queue will never hold more than 26 items (the letters A-Z), making the space requirement constant regardless of how massive the input array is.

In [5]:
import heapq
import collections
from typing import List

class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        # 1. Count the frequencies of each task
        # Counter(['A','A','B']) -> {'A': 2, 'B': 1}
        count = collections.Counter(tasks)
        
        # 2. Build the Max-Heap using the Negative Trick
        # We only care about the counts, not the letters themselves
        max_heap = [-c for c in count.values()]
        heapq.heapify(max_heap)
        
        # 3. The Cooldown Room: will store tuples of (remaining_count, available_time)
        queue = collections.deque()
        
        # The master clock
        time = 0
        
        # 4. Run the CPU simulation
        # The CPU keeps running as long as there is stuff in the heap OR the waiting room
        while max_heap or queue:
            time += 1
            
            # If there is a task ready to run, do it!
            if max_heap:
                # Pop the most frequent task and simulate running it (add 1 to reduce the negative value)
                current_count = heapq.heappop(max_heap) + 1
                
                # If the task still has runs left, send it to the cooldown room!
                if current_count != 0:
                    # It can leave the room at current `time` + `n`
                    queue.append((current_count, time + n))
            
            # Check the cooldown room: Is the person at the front ready to come out?
            if queue and queue[0][1] == time:
                ready_task_count, _ = queue.popleft()
                heapq.heappush(max_heap, ready_task_count)
                
        return time

### Problem 006: Design Twitter (LeetCode 355)

### Problem Definition and Constraints
Implement a simplified version of Twitter which allows users to post tweets, follow/unfollow each other, and view the 10 most recent tweets within their own news feed.
* `Twitter()`: Initializes the twitter object.
* `postTweet(userId, tweetId)`: Publish a new tweet.
* `getNewsFeed(userId)`: Fetches at most the 10 most recent tweet IDs in the user's news feed (from users they follow and themselves).
* `follow(followerId, followeeId)`: Follow a user.
* `unfollow(followerId, followeeId)`: Unfollow a user.

**Constraints:**
* 1 <= userId, followerId, followeeId <= 500
* 0 <= tweetId <= 10^4
* At most 30,000 calls will be made. A user cannot follow themselves explicitly in the inputs, but their own tweets must show up in their feed.

### Core Logic: The "Global Timer" and "K-Way Merge"
To fetch a news feed, a naive approach would dump all tweets from the user and their followees into a massive list, sort it, and return the top 10. That is an $O(N \log N)$ operation that wastes huge amounts of processing on old tweets.

Instead, we combine the **Negative Trick** with the **Divide and Conquer** logic from "Merge K Sorted Lists":
1. **The Global Timer:** We need a way to mathematically prove which tweet was posted first. We maintain a global `time` counter. Since we want the *most recent* tweets, and Python only has Min-Heaps, we apply the Negative Trick: every time a tweet is posted, we use a negative timestamp (e.g., `0`, `-1`, `-2`). The most recent tweets become the smallest negative numbers, bubbling perfectly to the top of the Min-Heap.
2. **The K-Way Merge:** Each individual user's list of tweets is already chronologically sorted. To find the 10 most recent tweets across $K$ different users, we don't need to look at all their tweets. We just look at the *single most recent tweet* from each of them. We push these top $K$ tweets into a Min-Heap. 
3. **The Replenishment:** When we pop the absolute most recent tweet from the heap, we look at who posted it, grab their *second* most recent tweet, and push it into the heap to replace the one we just took.

### Approach: Hash Maps + Priority Queue
1. **State:** Use `tweetMap` (maps `userId` to a list of `[time, tweetId]` tuples) and `followMap` (maps `userId` to a `set` of `followeeIds`).
2. **Post/Follow/Unfollow:** These are simple $O(1)$ Hash Map operations. 
3. **GetNewsFeed:**
   * Create an empty `minHeap` and a `res` list. Ensure the user follows themselves.
   * Iterate through the user's `followeeIds`. For each one, grab the very last tweet in their `tweetMap` list (the most recent one). Push a tuple into the heap: `(time, tweetId, followeeId, index_of_next_oldest_tweet)`.
   * Loop up to 10 times: Pop the top of the heap, append the `tweetId` to `res`. If that specific `followeeId` has older tweets left in their list (checked using the `index`), push the next oldest tweet into the heap.

* **Time Complexity:** 
  * `postTweet`, `follow`, `unfollow`: $O(1)$.
  * `getNewsFeed`: $O(K + 10 \log K)$ where $K$ is the number of followees. Building the initial heap takes $O(K)$. Extracting 10 tweets takes $O(10 \log K)$.
* **Space Complexity:** $O(U \cdot M + U \cdot F)$ where $U$ is total users, $M$ is max tweets per user, and $F$ is max followees. The heap itself only takes $O(K)$ space.

In [6]:
import collections
import heapq
from typing import List

class Twitter:

    def __init__(self):
        # The global timer. Decrementing it allows us to use Python's default Min-Heap 
        # to naturally bubble the most recent tweets (smallest negative numbers) to the top.
        self.count = 0
        
        # Maps userId -> list of [count, tweetId]
        self.tweetMap = collections.defaultdict(list)
        
        # Maps userId -> set of followeeIds
        self.followMap = collections.defaultdict(set)

    def postTweet(self, userId: int, tweetId: int) -> None:
        # Append the tweet with its global timestamp
        self.tweetMap[userId].append([self.count, tweetId])
        # Tick the clock forward (more negative)
        self.count -= 1

    def follow(self, followerId: int, followeeId: int) -> None:
        self.followMap[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        if followeeId in self.followMap[followerId]:
            self.followMap[followerId].remove(followeeId)

    def getNewsFeed(self, userId: int) -> List[int]:
        res = []
        minHeap = []
        
        # A user must always see their own tweets in their feed
        self.followMap[userId].add(userId)
        
        # 1. INITIALIZE THE HEAP (The Starting Lineup)
        # Grab the single most recent tweet from the user and everyone they follow
        for followeeId in self.followMap[userId]:
            if followeeId in self.tweetMap:
                # The most recent tweet is at the very end of their list
                index = len(self.tweetMap[followeeId]) - 1
                count, tweetId = self.tweetMap[followeeId][index]
                
                # Push: (timestamp, tweetId, who_posted_it, index_of_their_next_tweet)
                heapq.heappush(minHeap, [count, tweetId, followeeId, index - 1])
                
        # 2. EXTRACT THE TOP 10 (The Replenishment Loop)
        while minHeap and len(res) < 10:
            count, tweetId, followeeId, index = heapq.heappop(minHeap)
            res.append(tweetId)
            
            # If the user who posted this tweet has older tweets, 
            # push their next most recent tweet into the arena!
            if index >= 0:
                next_count, next_tweetId = self.tweetMap[followeeId][index]
                heapq.heappush(minHeap, [next_count, next_tweetId, followeeId, index - 1])
                
        return res


        """
        =========================================================
        SIMULATION NOTES: User 1 follows User 2
        =========================================================
        postTweet(1, 101): count = 0,  tweetMap[1] = [[0, 101]]
        postTweet(2, 201): count = -1, tweetMap[2] = [[-1, 201]]
        postTweet(1, 102): count = -2, tweetMap[1] = [[0, 101], [-2, 102]]
        postTweet(2, 202): count = -3, tweetMap[2] = [[-1, 201], [-3, 202]]
        
        getNewsFeed(1):
        - Followees: {1, 2}
        
        - INITIALIZE HEAP:
          User 1's most recent: [-2, 102]. Next index = 0.
          User 2's most recent: [-3, 202]. Next index = 0.
          Heap = [
            [-3, 202, 2, 0], 
            [-2, 102, 1, 0]
          ]
          
        - EXTRACT LOOP:
          1. Pop [-3, 202]. res = [202]. 
             User 2 has another tweet at index 0! 
             Push [-1, 201, 2, -1] into Heap.
             Heap = [[-2, 102, 1, 0], [-1, 201, 2, -1]]
             
          2. Pop [-2, 102]. res = [202, 102].
             User 1 has another tweet at index 0!
             Push [0, 101, 1, -1] into Heap.
             Heap = [[-1, 201, 2, -1], [0, 101, 1, -1]]
             
          3. Pop [-1, 201]. res = [202, 102, 201].
             User 2's index is -1. No more tweets to push.
             
          4. Pop [0, 101]. res = [202, 102, 201, 101].
             User 1's index is -1. No more tweets to push.
             
        Heap empty. Return [202, 102, 201, 101].
        """

### Problem 007: Find Median From Data Stream (LeetCode 295) [HARD]

### Problem Definition and Constraints
Design a data structure that allows you to continuously add numbers from a data stream and instantly find the median of the numbers seen so far.
* The median is the middle value in a sorted list. 
* If the list has an odd number of elements, the median is the exact middle element.
* If the list has an even number of elements, the median is the average of the two middle elements.

**Examples:**
* `addNum(1)` -> Stream is `[1]`. Median = `1.0`.
* `addNum(3)` -> Stream is `[1, 3]`. Median = `2.0`.
* `addNum(2)` -> Stream is `[1, 2, 3]`. Median = `2.0`.

* Constraints:
  * -100,000 <= num <= 100,000
  * At most 50,000 calls will be made.

### Core Logic: The "Two Halves" (Min-Heap & Max-Heap)
If you append numbers to an array and sort them every time `findMedian` is called, it will take $O(N \log N)$ time per call—which completely fails the efficiency constraints. 

To achieve $O(1)$ median lookups, we conceptually slice the sorted data perfectly in half.
1. **The Lower Half (`small`):** Contains the smaller 50% of the numbers. We want instant access to the *largest* number in this half, so we use a **Max-Heap**.
2. **The Upper Half (`large`):** Contains the larger 50% of the numbers. We want instant access to the *smallest* number in this half, so we use a **Min-Heap**.

The median is always resting right at the top of these two heaps! 

**The Two Golden Rules of the Heaps:**
1. **Order Rule:** Every number in `small` must be $\le$ every number in `large`. If the highest number in `small` accidentally exceeds the lowest number in `large`, we must immediately pop it from `small` and push it to `large`.
2. **Balance Rule:** The sizes of the two heaps can never differ by more than 1. If one heap gets too big, we pop its top element and push it onto the other heap.

### Approach: Two Heaps
1. Initialize `small` (Max-Heap, using the Python Negative Trick) and `large` (Min-Heap).
2. **`addNum(val)`:**
   * Always push the new number into `small` by default.
   * **Enforce Order:** If `small` and `large` both have elements, and the top of `small` is strictly greater than the top of `large`, pop from `small` and push to `large`.
   * **Enforce Balance:** If `len(small) > len(large) + 1`, pop from `small` and push to `large`. If `len(large) > len(small) + 1`, pop from `large` and push to `small`.
3. **`findMedian()`:**
   * If `small` has more elements, the median is the top of `small`.
   * If `large` has more elements, the median is the top of `large`.
   * If sizes are equal, the median is the average of the tops of both heaps.

* **Time Complexity:** $O(\log n)$ for `addNum()` because pushing and popping from heaps takes logarithmic time. $O(1)$ for `findMedian()` because peeking at the top of a heap is instant.
* **Space Complexity:** $O(n)$ — We store all numbers across the two heaps.

In [7]:
import heapq

class MedianFinder:

    def __init__(self):
        # The lower half of the numbers. 
        # Python only has Min-Heaps, so we use the Negative Trick to make it a Max-Heap.
        self.small = [] 
        
        # The upper half of the numbers. A standard Min-Heap.
        self.large = []

    def addNum(self, num: int) -> None:
        # 1. Always push to the small heap by default (don't forget the negative sign!)
        heapq.heappush(self.small, -num)
        
        # 2. ENFORCE ORDER RULE
        # The largest number in 'small' CANNOT be bigger than the smallest number in 'large'.
        # Since 'small' uses negative numbers, -self.small[0] gives the true positive max value.
        if self.small and self.large and (-self.small[0] > self.large[0]):
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
            
        # 3. ENFORCE BALANCE RULE
        # If the small heap gets too big, move an element to the large heap
        if len(self.small) > len(self.large) + 1:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
            
        # If the large heap gets too big, move an element to the small heap
        elif len(self.large) > len(self.small) + 1:
            val = heapq.heappop(self.large)
            heapq.heappush(self.small, -val)

    def findMedian(self) -> float:
        # If the lengths are uneven, the median is the top of the bigger heap
        if len(self.small) > len(self.large):
            return -self.small[0]
            
        if len(self.large) > len(self.small):
            return self.large[0]
            
        # If the lengths are exactly even, the median is the average of both tops
        return (-self.small[0] + self.large[0]) / 2.0


        """
        =========================================================
        SIMULATION NOTES: Stream = [1, 3, 2, 4]
        =========================================================
        
        addNum(1):
        - Push to small: small = [-1]
        - Order check: large is empty.
        - Balance check: small(1) vs large(0). Valid.
        - findMedian(): small is bigger -> returns 1.0.
        
        addNum(3):
        - Push to small: small = [-3, -1]
        - Order check: large is empty.
        - Balance check: small length is 2, large is 0. 
          small is too big! Pop -3 from small, push 3 to large.
          small = [-1], large = [3]
        - findMedian(): lengths are equal (1 and 1) -> (1 + 3) / 2.0 -> returns 2.0.
        
        addNum(2):
        - Push to small: small = [-2, -1]
        - Order check: top of small is 2. top of large is 3. 2 is NOT > 3. Valid.
        - Balance check: small(2) vs large(1). Valid.
        - findMedian(): small is bigger (2 vs 1) -> returns 2.0.
        
        addNum(4):
        - Push to small: small = [-4, -1, -2] (Max is 4)
        - Order check: top of small is 4. top of large is 3. 
          4 > 3! Rule broken! Pop 4 from small, push to large.
          small = [-2, -1], large = [3, 4]
        - Balance check: small(2) vs large(2). Valid.
        - findMedian(): lengths are equal -> (2 + 3) / 2.0 -> returns 2.5.
        """